# Week 3 — Prompting Strategies
**ComplianceGPT Lab · REU 2026 · Stony Brook University**

Week 2 answered *how* an LLM reads text. This notebook answers a more practical question: **given that Gemma3-4B still gets 8 of 137 real HIPAA cases wrong, can you fix any of them just by changing the prompt — without touching any code?**

You'll run three prompting strategies — **zero-shot**, **few-shot**, and **chain-of-thought** — against a real failing case from the actual GoldCoin experiment, using `gemma3:4b` running locally on your laptop via Ollama (the same model, same weights, that produced the 94.2% result you saw last week).

**What you'll build:**
- A working `call_ollama()` function you'll reuse all week
- Three prompt variants tested against a real, documented failure case
- A simplified fact-schema (6 fields, not the real system's 40) so you can focus on prompting technique, not schema memorization
- A repeatable pipeline you'll re-run on your **own** wrong prediction from Week 2

Run each cell with **Shift + Enter**. Post in `#python-help` on Slack if you're stuck — include your code and the full error message.

In [ ]:
# Run this cell first — checks Ollama is running and gemma3:4b is available
import requests, json, time

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "gemma3:4b"

try:
    tags = requests.get("http://localhost:11434/api/tags", timeout=3).json()
    names = [m["name"] for m in tags["models"]]
    print("Ollama is running ✓")
    print(f"{MODEL} available:", "✓" if MODEL in names else "✗ — run: ollama pull gemma3:4b")
except Exception as e:
    print("Ollama is NOT reachable. Start it with the Ollama app, or run `ollama serve` in a terminal.")
    print("Error:", e)


---
## The Case You're Fixing

This is **row_id 40** from the real Week 2 experiment (`final_vast_gemma3_4b.csv`) — a genuine false positive. The real system's 24-oracle extraction hallucinated **six** enabling facts that weren't actually supported by the text, and the formal engine (correctly, given those facts) returned PERMITTED. Ground truth is **DENIED**.

We're going to use a **simplified 6-field schema** instead of the real system's 40 fields — the goal this week is to learn prompting technique, not memorize the production schema. (You'll see the real one at the end of this notebook.)

In [ ]:
# The real scenario text, exactly as the model saw it (truncated at 300 characters — that's real too)
CASE_TEXT = (
    "In 2012, an individual found himself requiring emergency medical services and was "
    "taken by ambulance to a hospital. To receive this necessary medical attention, he provided the "
    "ambulance service, operated by the Philadelphia Fire Department Emergency Medical Services (EMS), "
    "with his personal and sensitive"
)

GROUND_TRUTH_VERDICT = "DENIED"          # what the case actually should resolve to
REAL_MODEL_VERDICT   = "PERMITTED"       # what Gemma3-4B actually said in the Week 2 run — WRONG

# The simplified schema for this exercise — 6 fields instead of the real system's 40
SCHEMA = '''{
  "sender_role": "hospital | ems | physician | other",
  "receiver_role": "patient | billing_company | law_enforcement | other",
  "purpose": "treatment | payment | law-enforcement | other",
  "is_business_associate": true or false,
  "is_required_by_law": true or false,
  "has_ba_agreement": true or false
}'''

print(CASE_TEXT)
print(f"\nGround truth: {GROUND_TRUTH_VERDICT}   |   Real Week 2 run said: {REAL_MODEL_VERDICT} (wrong)")


In [ ]:
# Your reusable LLM-calling function — you'll use this in every exercise below
def call_ollama(prompt, temperature=0.0, model=MODEL):
    """Send a prompt to a local Ollama model and return the raw text response."""
    r = requests.post(OLLAMA_URL, json={
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": temperature},
    }, timeout=120)
    r.raise_for_status()
    return r.json()["response"]


def extract_json(text):
    """Pull the first {...} block out of a model response and parse it. Returns None on failure."""
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1:
        return None
    try:
        return json.loads(text[start:end+1])
    except json.JSONDecodeError:
        return None


def simplified_verdict(fields):
    """Toy stand-in for the real Soufflé engine: if ANY enabling oracle fired, the
    (simplified) engine grants PERMITTED. This mirrors how the real 40-oracle system works —
    one wrongly-true oracle is enough to flip the verdict."""
    if fields is None:
        return "PARSE_ERROR"
    enabling = ["is_business_associate", "is_required_by_law", "has_ba_agreement"]
    return "PERMITTED" if any(fields.get(k) is True for k in enabling) else "DENIED"

print("Helper functions ready: call_ollama(), extract_json(), simplified_verdict()")


---
## Exercise 1 — Zero-Shot

**The concept:** give the model only the instructions and the schema. No examples. Fastest to write, fastest to iterate — and exactly what the real pipeline used to look like in its earliest version.

This one is fully worked for you — run it and read the output before moving to Exercise 2.

In [ ]:
zero_shot_prompt = f"""You are extracting facts from a HIPAA scenario. Read the scenario and output ONLY a JSON object matching this schema:
{SCHEMA}

Scenario: {CASE_TEXT}

JSON:"""

zero_shot_response = call_ollama(zero_shot_prompt)
zero_shot_fields = extract_json(zero_shot_response)

print("--- RAW RESPONSE ---")
print(zero_shot_response)
print("\n--- PARSED FIELDS ---")
print(zero_shot_fields)
print("\n--- SIMPLIFIED VERDICT ---")
print(simplified_verdict(zero_shot_fields), " (ground truth:", GROUND_TRUTH_VERDICT, ")")


**Expected output (example — Ollama sampling means yours may vary slightly, even at temperature 0):**
```json
{
  "sender_role": "ems",
  "receiver_role": "patient",
  "purpose": "treatment",
  "is_business_associate": false,
  "is_required_by_law": true,
  "has_ba_agreement": false
}
```
`is_required_by_law` comes back `true` with no textual basis for it — the same style of hallucination that caused the real failure. Simplified verdict: **PERMITTED** — still wrong. Zero-shot alone doesn't fix it.

---
## Exercise 2 — Few-Shot

**The concept:** show the model 2 worked examples *before* the real question. This is usually the single biggest lever for getting consistent output format and consistent judgment calls — the model pattern-matches against your examples, not just its own training data.

**Your task:** write two examples. Each needs a one-sentence scenario and a matching correctly-filled JSON object. Try to pick examples that are *similar in shape* to the failing case (an EMS/billing/business-relationship scenario) — that's what makes few-shot work, not just "any" example.

**Hints:**
- Example 1 could show a *treatment* scenario where none of the three enabling fields should be true
- Example 2 could show a *real* business-associate scenario (a signed vendor contract) where `is_business_associate` and `has_ba_agreement` genuinely should be true — showing the model what the *positive* case actually looks like

In [ ]:
# TODO: fill in EXAMPLE_1 and EXAMPLE_2 below. Each must be a valid JSON object matching SCHEMA.

EXAMPLE_1_SCENARIO = None   # replace with a one-sentence scenario string
EXAMPLE_1_JSON      = None  # replace with a JSON string matching SCHEMA, e.g. '{"sender_role": "physician", ...}'

EXAMPLE_2_SCENARIO = None   # replace with a one-sentence scenario string
EXAMPLE_2_JSON      = None  # replace with a JSON string matching SCHEMA

few_shot_prompt = f"""You are extracting facts from a HIPAA scenario. Output ONLY a JSON object matching this schema:
{SCHEMA}

Example 1:
Scenario: {EXAMPLE_1_SCENARIO}
JSON: {EXAMPLE_1_JSON}

Example 2:
Scenario: {EXAMPLE_2_SCENARIO}
JSON: {EXAMPLE_2_JSON}

Now extract:
Scenario: {CASE_TEXT}
JSON:"""

few_shot_response = call_ollama(few_shot_prompt)
few_shot_fields = extract_json(few_shot_response)

print(few_shot_response)
print("\nSimplified verdict:", simplified_verdict(few_shot_fields), " (ground truth:", GROUND_TRUTH_VERDICT, ")")


---
## Exercise 3 — Chain-of-Thought

**The concept:** ask the model to reason step-by-step *before* it commits to an answer. This spends more tokens (and time) but often catches exactly the kind of ungrounded inference we saw in Exercise 1 — because the model has to point to specific textual evidence for each field instead of pattern-matching the whole scenario at once.

**Your task:** write 4–5 numbered reasoning steps the model should walk through before answering. Steer it toward requiring *explicit textual evidence* for each of the three enabling fields — that's the actual fix for this failure mode.

In [ ]:
# TODO: write your own step-by-step reasoning instructions below (replace the ... )

cot_steps = """
1. ...
2. ...
3. ...
4. ...
"""

cot_prompt = f"""You are extracting facts from a HIPAA scenario. Think step by step, then output a JSON object matching this schema:
{SCHEMA}

Scenario: {CASE_TEXT}

Think step by step:
{cot_steps}

Then output the JSON on its own final line, prefixed with JSON:"""

cot_response = call_ollama(cot_prompt)
cot_fields = extract_json(cot_response)

print(cot_response)
print("\nSimplified verdict:", simplified_verdict(cot_fields), " (ground truth:", GROUND_TRUTH_VERDICT, ")")


---
## Exercise 4 — Compare All Three

**Your task:** build a small comparison table: strategy name → simplified verdict → correct? Use the three `*_fields` variables from Exercises 1–3.

In [ ]:
results = {
    "Zero-shot": zero_shot_fields,
    "Few-shot": few_shot_fields,
    "Chain-of-thought": cot_fields,
}

print(f"{'Strategy':<18} {'Verdict':<12} {'Correct?'}")
print("-" * 42)
for name, fields in results.items():
    # TODO: compute verdict using simplified_verdict(fields)
    # TODO: compute whether verdict == GROUND_TRUTH_VERDICT
    verdict = None
    correct = None
    print(f"{name:<18} {str(verdict):<12} {correct}")


**Reflection (double-click to edit):**

Did any of the three strategies produce the correct DENIED verdict? If none did, which one got *closest* (fewest wrongly-true enabling fields)? Look back at the real production prompt's instruction #11 in `connector/llm1_extractor.py` (you'll see it at the end of this notebook) — what does it do differently from your zero-shot prompt that might explain the gap?

_Your answer:_


---
## Exercise 5 — Does Temperature Matter? (bonus)

**The concept:** Week 2 told you to always use `temperature=0` for extraction. Let's verify that's not just folklore — run the *same* zero-shot prompt 3 times at `temperature=0` and 3 times at `temperature=0.9`, and see which one is actually reproducible.

In [ ]:
# TODO: call zero_shot_prompt 3 times at temperature=0 and 3 times at temperature=0.9
# Print each response's parsed fields. Are the temperature=0 runs identical? Are the 0.9 runs identical?

for temp in [0.0, 0.9]:
    print(f"=== temperature={temp} ===")
    for i in range(3):
        response = None   # replace with: call_ollama(zero_shot_prompt, temperature=temp)
        fields   = None   # replace with: extract_json(response)
        print(f"  run {i+1}:", fields)
    print()


---
## Exercise 6 — Now Fix Your Own Case

Load your **own** Week 2 cluster results CSV (from Exercise 9 last week) and pick one row where `match == "N"` — a case *you* got wrong. Repeat Exercises 1–4 on it.

If you don't have your own cluster CSV yet, use the shared one at
`/Users/priscilladanso/Documents/GitHub/COMPLIANCEGPT/experiments/finalserverrun/final_vast_gemma3_4b.csv` and pick any `row_id` other than 40 where `match == "N"` (real failing row_ids: 16, 27, 30, 46, 59, 66, 123).

In [ ]:
import pandas as pd

# TODO: update this path to your own Week 2 results CSV, or use the shared one described above
RESULTS_CSV = "PASTE_YOUR_CSV_PATH_HERE.csv"

# TODO:
# 1. df = pd.read_csv(RESULTS_CSV)
# 2. failures = df[df["match"] == "N"]
# 3. pick one row: my_case = failures.iloc[0]   (or any index you like)
# 4. print(my_case["question"], my_case["ground_truth"])
# 5. write your own 6-field (or fewer) SCHEMA relevant to what went wrong in this case
# 6. re-run Exercises 1–4 using my_case's text instead of CASE_TEXT

df = None
failures = None
my_case = None


---
## Where This Goes Next

The real extraction prompt — `connector/llm1_extractor.py` in the COMPLIANCEGPT repo — is not a 6-field schema like the one you used today. It's a **40-field schema with ~2,000 lines of instructions**: explicit trigger words for each oracle, worked examples for edge cases, and 14 numbered rules for exactly the kind of ambiguity you just fought through in Exercises 2 and 3 (see rule #11 — "EVIDENCE PREDICATES FOR JUDICIAL/LEGAL PROCESS" — which exists *specifically* because of hallucinations like the one in Exercise 1).

Every rule in that file was added because a real case, just like row 40, broke a simpler version of the prompt. That's the process you just did in miniature.

**Wednesday in class:** open `connector/llm1_extractor.py` (lines 87–270) together and match each numbered rule to the failure mode it exists to prevent.